In [ ]:
import os
import logging
import time
import torch
logging.basicConfig(level=logging.ERROR)

In [ ]:
google_drive_mountpoint = "/content/drive"
sourcecode_url = "https://github.com/stonebo/Research-Estimator-xMem.git"
default_branch = "feat/llm"
models = [
	"deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
	"Qwen/Qwen3-4B",
	"meta-llama/Llama-3.2-3B-Instruct",
	"JetBrains/Mellum-4b-base",
]
model = models[-1]
opt = "Adafactor"
bs = 1
gpu_id = 0
fp16 = False

In [ ]:
colab_enable = True if "COLAB_RELEASE_TAG" in os.environ else False
if colab_enable:
    from google.colab import drive, userdata
    from urllib.parse import urlparse
    from pathlib import Path
    dir_name = "repo-xmem"
    repo_url = urlparse(sourcecode_url)
    # prepare repo
    if not Path(os.getcwd()).joinpath(dir_name).is_dir():
      repo_url = f"https://{userdata.get('G_USER')}:{userdata.get('G_PAT')}@{repo_url.hostname}{repo_url.path}"
      !apt install git
      !git clone {repo_url} {dir_name}
    # change workdir
    %cd {dir_name}
    !git fetch origin
    !git checkout {default_branch}
    # install dependencies
    if Path(os.getcwd()).joinpath("requirement-r.txt").is_file():
        !pip install -r requirement.txt
    else:
        raise FileNotFoundError(f"missing requirement.txt file")
    # mount google drive
    if not Path(google_drive_mountpoint).is_dir():
        drive.mount(google_drive_mountpoint)


In [ ]:
from exp.run import ExperimentRun, SummarySectionName
from exp.config import LargeTransformerExperiments
config = LargeTransformerExperiments()

if colab_enable:
	output_dir = Path().home().joinpath(config.run_id)
	target_dir = Path(google_drive_mountpoint).joinpath("MyDrive/100-ResearchData/2025-Middleware-xMem LLM/005-CoLab", config.run_id)
	if target_dir.is_dir() is False:
		target_dir.mkdir(parents=True, exist_ok=True)
	# create a softlink for output dir
	output_dir.symlink_to(target_dir, target_is_directory=True)


In [ ]:
config.debug = False
config.repeats = 1
config.gpu_id = gpu_id
config.fp16 = fp16
exp = ExperimentRun(config=config)


In [ ]:
exp.add_task(
	model_name=model,
	batch_size=bs,
	optimizer=opt,
	gpu_id=gpu_id
)

In [ ]:
exp.run_group_truth()
time.sleep(2)
torch.cuda.empty_cache()
time.sleep(2)


In [ ]:
result = exp.run_estimation(estimators=[SummarySectionName.solution])